In [ ]:
import matplotlib.pyplot as plt
from matplotlib.text import Text
import numpy as np
import glob
import pandas as pd
import os
from datetime import datetime, timezone, timedelta

In [ ]:
calibration_frequency = 410

data_dir = f"/home/scratch/dbautist/TEST/{calibration_frequency}/"

csvs = glob.glob(f"{data_dir}/*/*.csv")
csvs.sort()

In [ ]:
# exclude the following days due to extreme RFI 
to_exclude = ["2025_078", "2024_291", "2024_292"]
for bad_day in to_exclude:
    try:
        csvs.remove(f"/home/scratch/dbautist/TEST/410/{bad_day}/{bad_day}.csv")
    except:
        pass

In [ ]:
full_dates = []
all_dates = []

first_day = os.path.basename(csvs[0])[:-4]
latest_day = os.path.basename(csvs[-1])[:-4]

date_difference = datetime.strptime(latest_day, "%Y_%j") - datetime.strptime(first_day, "%Y_%j")
n_days = date_difference.days
one_day = timedelta(days=1)

for i in range(n_days+1):
    full_dates.append((datetime.strptime(first_day, "%Y_%j") + i*one_day).strftime("%Y_%j"))
    all_dates.append((datetime.strptime(first_day, "%Y_%j") + i*one_day))

In [ ]:
timestamps = []
# full_dates = []
# years = []
for i in range(len(csvs)):
    timestamps.append(int(os.path.basename(csvs[i]).replace(".csv", "")[-3::]))

# bad calibrations
mean_flux = []
date = []

mask = []
grid = np.empty((len(full_dates), 1024))
missing_dates = []
for i in range(len(full_dates)):
    year = full_dates[i][:4]
    date_str = full_dates[i][-3:]
    csv_path = f"{data_dir}/{year}_{date_str}/{year}_{date_str}.csv".replace("//", "/")
    if csv_path in csvs:#os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        grid[i] = np.array(df["intensity"], dtype=float)
        mask.append(np.zeros(1024))

        # bookkeeping
        mean_flux.append(np.nanmean(df["intensity"]))
        date.append(f"{year}_{date_str}")
    else:
        print(csv_path)
        grid[i] = np.empty(1024)
        mask.append(np.ones(1024))
        missing_dates.append(f"{year}_{date_str}")
        print("no data... masking day")

masked_grid = np.ma.masked_array(grid, mask=mask)

In [ ]:
from datetime import datetime
human_readable_dates = []
for date in full_dates:
    time_obj = datetime.strptime(date, "%Y_%j")
    MM_DD_YY = time_obj.strftime("%m-%d-%y")
    human_readable_dates.append(MM_DD_YY)

In [ ]:
unit = "Jy"
human_readable = True

frequency = df["frequency"]
average_spectrum = np.nanmean(masked_grid, axis=0)
time_series = np.nanmean(masked_grid, axis=1)

extent = [400, 800, min(all_dates), max(all_dates)]

height = 10
ratio = 3
fig = plt.figure(figsize=(10,height))
gs = fig.add_gridspec(2,2, hspace=0.02, wspace=0.03, width_ratios=[ratio,1], height_ratios=[1,ratio])
(ax1, ax2), (ax3, ax4) = gs.subplots(sharex="col", sharey="row")
ax1.set_title(f"GBO CHIME outrigger data calibrated at {calibration_frequency} MHz")
ax1.semilogy(frequency, average_spectrum, color="black", linewidth=1)
ax1.set_ylabel(f"average flux [{unit}]")
ax2.set_visible(not ax2)
wf = ax3.imshow(masked_grid, aspect="auto", extent=extent, origin="lower", vmin=1e4, vmax=7e5, interpolation="none")#vmin=0, vmax=4e7,)
ax3.set_xlabel("Frequency [MHz]")
ax3.set_ylabel("date [YYYY_ddd]")
ax3.set_yticks(all_dates)
if human_readable:
    ax3.set_yticklabels(human_readable_dates)
    ax3.set_ylabel("date [MM-DD-YYYY]")
else:
    ax3.set_yticklabels(full_dates)
    ax3.set_ylabel("date [YYYY_ddd]")
count = 0
separator_index = len(full_dates)//10
if separator_index == 0:
    separator_index = 1
print("total tick labels:", len(ax3.get_yticklabels()))
total_ticks = len(ax3.get_yticklabels())
for i,label in enumerate(ax3.get_yticklabels()):
    # print(count)
    too_close = (total_ticks - i) < int(0.05 * total_ticks)
    if (count%separator_index != 0 or too_close) and count != (total_ticks - 1):
        label.set_visible(False)
    count += 1

ax4.plot(time_series, all_dates, color="black", linewidth=1)
ax4.set_xlabel(f"avg power\nper channel\n[{unit}]")
ax4.set_xscale("log")
# plt.savefig(f"/users/dbautist/AAS_poster/calibrated_{calibration_frequency}.png", bbox_inches='tight', transparent=False)#f"/users/dbautist/CHIME_landing_directory/confluence/calibrated_{calibration_frequency}.png", bbox_inches='tight', transparent=False)
# plt.savefig(f"/users/dbautist/AAS_poster/calibrated_{calibration_frequency}.pdf", bbox_inches='tight', transparent=False)
fig.colorbar(wf, ax=ax4, label=f'flux [{unit}]', location='right')
plt.show()

# Fitting a period to the data variation

In [ ]:
from scipy.optimize import curve_fit

def wave(x, P, phi, A, B):
    yy = A * np.cos(2*np.pi/P*x + phi) + B
    return yy

In [ ]:
xx = np.arange(len(time_series))
p0 = (365,1,1e6,0.5e6)
bounds = ((100, 0, 0.1e6, 0), 
          (700, np.inf, 0.75e6, 0.5e6)
         )

plt.figure()

# setting uncertainties
uncertainties = np.ones_like(xx)*1e5
# roc = np.diff(time_series)
# dy_mask = (np.abs(roc) > 0.14e6).data
# uncertainties[1:][dy_mask] = 1e5
uncertainties[200:400] = 1e6
uncertainties[600:770] = 1e6

plt.axvspan(200, 400, alpha=0.25, color="red", label='high uncertainty')
plt.axvspan(600, 770, alpha=0.25, color="red")
# =================================== #

coef_letters = ["P [d]" , "phi", "A", "B"]

coeff, cov = curve_fit(wave, xx[~time_series.mask], time_series.data[~time_series.mask], 
                       p0=p0,
                       sigma=uncertainties[~time_series.mask]
                       ) # bounds=bounds)

label_string = ""
for i in range(len(coef_letters)):
    label_string += coef_letters[i] + ": " + str(np.round(coeff[i], decimals=3)) + '\n'


plt.plot(xx, time_series)

plt.plot(xx, wave(xx, *coeff), label=f"fit coeff:\n{label_string}")
# plt.yscale("log")
plt.ylim(0, 0.3e7)
plt.legend()
plt.xlabel("days (arbitrary start)")
plt.ylabel(f"avg power\nper channel\n[{unit}]")

In [ ]:
len(xx)

In [ ]:
coef_letters = ["P [d]" , "phi", "A", "B"]

for i in range(len(coef_letters)):
    print(coef_letters[i] + "\t: " + str(np.round(coeff[i], decimals=3)) + "\t+/- " + str(np.round(cov[i,i], decimals=3)))

In [ ]:
residuals = time_series - wave(xx, *coeff)

plt.figure()
plt.title("Residuals")
plt.plot(xx, residuals, color="red")
plt.hlines(0, xx.min(), xx.max(), color="black")
plt.xlabel("days (arbitrary start)")
plt.ylabel(f"avg power\nper channel\n[{unit}]")
plt.ylim(-1e6, 1e6)
plt.grid()

In [ ]:
sd = np.std(residuals)
print(sd)
np.sum((residuals / sd) > 1)

In [ ]:
roc = np.diff(time_series)
limit = 0.14e6

plt.figure()
plt.plot(xx[1:], roc)
plt.ylim(-1e6, 1e6)
plt.hlines(-limit, xx.min(), xx.max(), color="red")
plt.hlines(limit, xx.min(), xx.max() , color="red")
plt.grid()